# ПЗ-3. Мини-LLaVA: обучение проектора

## 1. Цель
Обучить только MLP, связывающий замороженный SigLIP с замороженной Qwen.

## 2. Датасеты и модели
COCO Captions 2017: 20000 уникальных изображений train с одной подписью и 500 изображений val со всеми эталонными подписями. Один проход по train для каждого проектора. Подписи COCO английские, заданный промпт русский; язык целевых подписей не подменяется переводом.

[SigLIP](https://huggingface.co/google/siglip-base-patch16-224), [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).

Источник задания: `Практика_06_11_2026_1_week.docx`. Приложения А.1–А.7 отсутствуют; ниже приведены стандартные реализации. LoRA-проект не входит в эти шесть работ.

## 3. Подготовка окружения
Откройте этот notebook в Google Colab. Для ПЗ-1 выберите CPU, для ПЗ-3–6 — T4; для ПЗ-2 GPU ускорит получение признаков. Выполните установку, перезапустите сеанс, затем выполните остальные ячейки сверху вниз. Все результаты сохраняются в папке задания; скачайте её вместе с ноутбуком. Данные и веса кэшируются в соседней папке `data`. При переносе в Colab прежние результаты доступны только после загрузки файлов. GPU использует fp16. Числа в сохранённых выводах относятся только к реально выполненным ячейкам. Ячейки без execution_count не запускались.

In [ ]:
%pip install torch==2.9.1 torchvision==0.24.1 transformers==4.57.6 numpy==2.3.5 pandas==2.3.3 matplotlib==3.10.8 pillow==12.0.0 datasets==4.4.1 accelerate==1.12.0 sentencepiece==0.2.1 scikit-learn==1.7.2 pycocoevalcap==1.2 reportlab==4.4.5 huggingface-hub==0.36.2 safetensors==0.8.0 tokenizers==0.22.2 protobuf==6.33.2 ftfy==6.3.1 requests==2.34.2 fsspec==2025.10.0 pyarrow==26.0.1

In [1]:
LAB = 'PZ_3'
REVISIONS = {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'lmms-lab-encoder/RefCOCO': 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'}

import os
import sys
import json
import random
import time
import math
import gc
import hashlib
import platform
import importlib.metadata
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd() if Path.cwd().name == LAB else Path.cwd() / LAB
ROOT.mkdir(exist_ok=True)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
DATA = ROOT.parent / "data"
DATA.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(DATA / "huggingface")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

def save_json(name, value):
    path = OUT / name
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def save_figure(name):
    plt.savefig(OUT / name, dpi=130, bbox_inches="tight")
    plt.show()

versions = {p.metadata["Name"]: p.version for p in importlib.metadata.distributions() if p.metadata["Name"]}
environment = {"python": sys.version, "platform": platform.platform(), "seed": SEED,
               "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
               "versions": versions, "revisions": REVISIONS}
save_json("environment.json", environment)
print({k: v for k, v in environment.items() if k != "versions"})

C:\Users\darri\PycharmProjects\cv061126\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'seed': 42, 'device': 'cpu', 'gpu': None, 'revisions': {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'lmms-lab-encoder/RefCOCO': 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'}}


In [2]:
from xml.sax.saxutils import escape
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as PDFImage
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from matplotlib import font_manager

def write_report(title, goal, work, findings, table=None, figure=None):
    pdfmetrics.registerFont(TTFont("Student", font_manager.findfont("DejaVu Sans")))
    style = ParagraphStyle("Student", fontName="Student", fontSize=10, leading=14, spaceAfter=8)
    heading = ParagraphStyle("Heading", parent=style, fontSize=14, leading=18, spaceAfter=12)
    story = [Paragraph(escape(title), heading), Paragraph(escape("Цель. " + goal), style),
             Paragraph(escape("Что было сделано. " + work), style)]
    for text in findings:
        story.append(Paragraph(escape(text), style))
    if table is not None and len(table):
        rows = [[Paragraph(escape(str(x)), style) for x in table.columns]]
        rows += [[Paragraph(escape(f"{x:.4f}" if isinstance(x, float) else str(x)), style) for x in row]
                 for row in table.itertuples(index=False, name=None)]
        item = Table(rows, repeatRows=1, colWidths=[495 / len(table.columns)] * len(table.columns))
        item.setStyle(TableStyle([("VALIGN", (0,0), (-1,-1), "TOP"),
                                 ("GRID", (0,0), (-1,-1), 0.3, colors.grey)]))
        story.extend([item, Spacer(1, 10)])
    if figure is not None and Path(figure).exists():
        with Image.open(figure) as im:
            ratio = im.height / im.width
        story.append(PDFImage(str(figure), width=460, height=min(250, 460 * ratio)))
    story.append(Paragraph(escape("Индивидуальный вклад. Подготовлены реализации, протоколы экспериментов, тесты и оформление. Код и текст подготовлены с помощью ИИ-ассистента; фактические запуски и ограничения перечислены выше. Авторство невыполненных опытов не заявляется."), style))
    story.append(Paragraph(escape("Воспроизведение: notebook " + LAB + ".ipynb; seed=42. Версии среды и точные ревизии весов сохранены в results/environment.json. Контрольные вопросы разобраны в ноутбуке."), style))
    SimpleDocTemplate(str(ROOT / "report.pdf"), pagesize=(595,842), rightMargin=50, leftMargin=50,
                      topMargin=45, bottomMargin=45).build(story)
    print("PDF:", ROOT / "report.pdf")

In [3]:
import requests
from zipfile import ZipFile

def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(4):
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                with temporary.open("wb") as stream:
                    for block in response.iter_content(1024 * 1024):
                        stream.write(block)
            temporary.replace(destination)
            return destination
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(attempt + 1)

def coco_annotations(year=2017):
    archive = download(f"https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval{year}.zip",
                       DATA / f"annotations_trainval{year}.zip")
    with ZipFile(archive) as z:
        for name in z.namelist():
            if name.startswith("annotations/") and name.endswith(".json"):
                target = DATA / name
                if not target.exists():
                    target.parent.mkdir(exist_ok=True)
                    target.write_bytes(z.read(name))
    manifest = {"url": f"https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval{year}.zip",
                "sha256": hashlib.file_digest(archive.open("rb"), "sha256").hexdigest()}
    save_json(f"coco{year}_source.json", manifest)

def caption_pairs(split, count, seed=SEED):
    data = json.loads((DATA / "annotations" / f"captions_{split}2017.json").read_text())
    captions = defaultdict(list)
    for ann in data["annotations"]:
        captions[ann["image_id"]].append(ann)
    ids = sorted(captions)
    rng = random.Random(seed)
    rng.shuffle(ids)
    selected = ids[:count]
    assert len(selected) == count
    records = []
    for image_id in selected:
        refs = sorted(captions[image_id], key=lambda a: a["id"])
        target = rng.choice(refs)
        records.append({"image_id": image_id, "caption_id": target["id"], "caption": target["caption"],
                        "references": [r["caption"] for r in refs], "split": split})
    return records

def coco_image(record, year=2017):
    split = record["split"]
    image_id = record["image_id"]
    name = f"{image_id:012d}.jpg" if year == 2017 else f"COCO_{split}{year}_{image_id:012d}.jpg"
    path = download(f"https://s3.amazonaws.com/images.cocodataset.org/{split}{year}/{name}", DATA / f"{split}{year}" / name)
    with Image.open(path) as image:
        return image.convert("RGB")

def load_images(records):
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(coco_image, records))

## 4. Реализация
Слияние 2×2 — среднее четырёх соседних пространственных признаков. Размерность входа MLP сохраняется. В обеих версиях MLP имеет два Linear с GELU. Labels не сдвигаются вручную: causal LM сопоставляет logits[:, :-1] с labels[:, 1:]. Последний токен промпта должен предсказывать первый токен подписи. Padding маскируется по длине, а не по значению токена: pad и eos у Qwen могут совпадать.

In [4]:
def merge_patches(features, n_tokens):
    if n_tokens == 196:
        return features
    if n_tokens != 49 or features.shape[1] != 196:
        raise ValueError("Ожидались 196 патчей и выход 49 или 196")
    b, _, d = features.shape
    return features.reshape(b,7,2,7,2,d).mean(dim=(2,4)).reshape(b,49,d)

def projector(input_dim, hidden_dim):
    return nn.Sequential(nn.Linear(input_dim,hidden_dim),nn.GELU(),nn.Linear(hidden_dim,hidden_dim))

def assemble_inputs(visual, prompt_ids, caption_ids, eos_id, pad_id, embedding):
    n, v, _ = visual.shape
    captions = [ids[:47]+[eos_id] for ids in caption_ids]
    lengths = [len(prompt_ids)+len(ids) for ids in captions]
    tokens = torch.full((n,max(lengths)),pad_id,dtype=torch.long,device=visual.device)
    attention = torch.zeros((n,v+max(lengths)),dtype=torch.long,device=visual.device)
    labels = torch.full_like(attention,-100)
    for i, ids in enumerate(captions):
        sequence = prompt_ids+ids
        tokens[i,:len(sequence)] = torch.tensor(sequence,device=visual.device)
        attention[i,:v+len(sequence)] = 1
        labels[i,v+len(prompt_ids):v+len(sequence)] = torch.tensor(ids,device=visual.device)
    text_embeddings = embedding(tokens)
    combined = torch.cat([visual.to(text_embeddings.dtype),text_embeddings],dim=1)
    return {"inputs_embeds":combined,"attention_mask":attention,"labels":labels}

In [5]:
from transformers import Qwen2Config, Qwen2ForCausalLM
torch.manual_seed(SEED)
tiny_config = Qwen2Config(vocab_size=64,hidden_size=32,intermediate_size=64,
                         num_hidden_layers=1,num_attention_heads=4,num_key_value_heads=2,
                         max_position_embeddings=512,attention_dropout=0.)
tiny_llm = Qwen2ForCausalLM(tiny_config).eval().requires_grad_(False)
visual_test = torch.randn(2,49,32,requires_grad=True)
packed = assemble_inputs(visual_test,[3,4],[[5,6,7],[8]],2,2,tiny_llm.get_input_embeddings())
assert (packed["labels"][:,:51] == -100).all()
assert packed["labels"][0,51:].tolist() == [5,6,7,2]
assert packed["labels"][1,51:].tolist() == [8,2,-100,-100]
assert packed["attention_mask"][1,-2:].sum() == 0
result = tiny_llm(**packed)
manual = F.cross_entropy(result.logits[:,:-1].contiguous().view(-1,64),packed["labels"][:,1:].contiguous().view(-1),ignore_index=-100)
assert torch.allclose(result.loss,manual,atol=1e-6)
result.loss.backward()
assert visual_test.grad is not None and visual_test.grad.abs().sum() > 0
assert all(p.grad is None for p in tiny_llm.parameters())
grid = torch.arange(196.).reshape(1,196,1)
pooled = merge_patches(grid,49)
assert pooled.shape == (1,49,1) and pooled[0,0,0] == (0+1+14+15)/4
unit_results = {"causal_loss":result.loss.item(),"manual_loss":manual.item(),
                "visual_gradient_norm":visual_test.grad.norm().item(),"mask_and_shift":True,"pooling":True,"passed":True}
save_json("unit_tests.json",unit_results)
display(unit_results)
del tiny_llm

{'causal_loss': 4.163172245025635,
 'manual_loss': 4.163172245025635,
 'visual_gradient_norm': 0.0046247439458966255,
 'mask_and_shift': True,
 'pooling': True,
 'passed': True}

## 5. Эксперименты
Полный запуск требует T4 и около 7 ГБ для кэша patch-признаков. Используются fp16 autocast и GradScaler, AdamW lr=1e-3, warmup 3%, batch=4 и накопление 8 микробатчей (эффективный batch=32). Сравнивается одна эпоха, одинаковые порядок данных и начальные веса MLP. Encoder не входит в измеренное время обучения проектора; время его предвычисления записано отдельно.

In [6]:
coco_annotations()
train_pairs = caption_pairs("train",20000)
validation_pairs = caption_pairs("val",500)
assert not ({r["image_id"] for r in train_pairs} & {r["image_id"] for r in validation_pairs})
save_json("pairs_train.json",train_pairs)
save_json("pairs_val.json",validation_pairs)

In [ ]:
from transformers import AutoImageProcessor, SiglipVisionModel, AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
if DEVICE.type != "cuda":
    raise RuntimeError("Полное обучение на 20000 парах требует CUDA GPU; запустите этот notebook на T4.")
vision_id = "google/siglip-base-patch16-224"
vision_processor = AutoImageProcessor.from_pretrained(vision_id,revision=REVISIONS[vision_id],use_fast=False)
vision = SiglipVisionModel.from_pretrained(vision_id,revision=REVISIONS[vision_id],torch_dtype=torch.float16).to(DEVICE).eval().requires_grad_(False)
cache_times = {}
for split, records in [("train",train_pairs),("val",validation_pairs)]:
    path = OUT/f"patch_features_{split}.npy"
    features = np.lib.format.open_memmap(path,mode="w+",dtype=np.float16,shape=(len(records),196,vision.config.hidden_size))
    start_time = time.perf_counter()
    for begin in tqdm(range(0,len(records),32),desc=f"SigLIP {split}"):
        images = load_images(records[begin:begin+32])
        pixels = vision_processor(images=images,return_tensors="pt").pixel_values.to(DEVICE,torch.float16)
        with torch.inference_mode():
            hidden = vision(pixel_values=pixels).last_hidden_state
        assert hidden.shape[1] == 196
        features[begin:begin+len(images)] = hidden.cpu().numpy()
    features.flush()
    cache_times[split] = time.perf_counter()-start_time
    del features
save_json("feature_cache.json",{"seconds":cache_times,"vision_revision":REVISIONS[vision_id],"train":20000,"val":500})
del vision
gc.collect()
torch.cuda.empty_cache()
language_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(language_id,revision=REVISIONS[language_id])
llm = AutoModelForCausalLM.from_pretrained(language_id,revision=REVISIONS[language_id],torch_dtype=torch.float16,
                                         attn_implementation="sdpa").to(DEVICE).requires_grad_(False)
llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
llm.config.use_cache = False
prompt_ids = tokenizer("Опиши изображение.",add_special_tokens=False).input_ids
caption_ids = [tokenizer(r["caption"],add_special_tokens=False).input_ids for r in train_pairs]
train_features = np.load(OUT/"patch_features_train.npy",mmap_mode="r")
val_features = np.load(OUT/"patch_features_val.npy",mmap_mode="r")

In [ ]:
training_rows, timing_rows = [], []
batch_size, accumulation, epochs = 4, 8, 1
num_batches = math.ceil(len(train_pairs)/batch_size)
total_steps = epochs*math.ceil(num_batches/accumulation)
for n_tokens in [196,49]:
    torch.manual_seed(SEED)
    mlp = projector(train_features.shape[-1],llm.config.hidden_size).to(DEVICE)
    optimizer = torch.optim.AdamW(mlp.parameters(),lr=1e-3,weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer,max(1,int(total_steps*0.03)),total_steps)
    scaler = torch.amp.GradScaler("cuda")
    llm.train()
    mlp.train()
    generator = torch.Generator().manual_seed(SEED)
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    start_time = time.perf_counter()
    updates, skipped = 0, 0
    for epoch in range(epochs):
        order = torch.randperm(len(train_pairs),generator=generator).tolist()
        for batch_index, begin in enumerate(tqdm(range(0,len(order),batch_size),desc=f"train {n_tokens}")):
            ids = order[begin:begin+batch_size]
            raw = torch.from_numpy(np.array(train_features[ids],copy=True)).to(DEVICE)
            group_size = min(accumulation,num_batches-(batch_index//accumulation)*accumulation)
            with torch.autocast("cuda",dtype=torch.float16):
                visual = mlp(merge_patches(raw,n_tokens))
                inputs = assemble_inputs(visual,prompt_ids,[caption_ids[i] for i in ids],
                                         tokenizer.eos_token_id,tokenizer.pad_token_id,llm.get_input_embeddings())
                loss = llm(**inputs,use_cache=False).loss
            if not torch.isfinite(loss):
                raise FloatingPointError("Неконечный training loss")
            scaler.scale(loss/group_size).backward()
            training_rows.append({"visual_tokens":n_tokens,"epoch":epoch+1,"microbatch":batch_index+1,
                                  "loss":loss.item(),"context_length":inputs["inputs_embeds"].shape[1],
                                  "lr":optimizer.param_groups[0]["lr"]})
            if (batch_index+1)%accumulation == 0 or batch_index+1 == num_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(mlp.parameters(),1.)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                    updates += 1
                else:
                    skipped += 1
                optimizer.zero_grad(set_to_none=True)
        torch.save({"state_dict":mlp.state_dict(),"tokens":n_tokens,"epochs":epoch+1,
                    "seed":SEED,"revisions":REVISIONS},OUT/f"projector_{n_tokens}.pt")
        pd.DataFrame(training_rows).to_csv(OUT/"training.csv",index=False)
    torch.cuda.synchronize()
    timing_rows.append({"visual_tokens":n_tokens,"seconds":time.perf_counter()-start_time,
                        "epochs":epochs,"optimizer_steps":updates,"skipped_steps":skipped,
                        "max_context":n_tokens+len(prompt_ids)+48})
    pd.DataFrame(timing_rows).to_csv(OUT/"timing.csv",index=False)
    del mlp, optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
llm.eval()
llm.gradient_checkpointing_disable()
llm.config.use_cache = True
generation_rows = []
for n_tokens in [196,49]:
    mlp = projector(val_features.shape[-1],llm.config.hidden_size).to(DEVICE)
    mlp.load_state_dict(torch.load(OUT/f"projector_{n_tokens}.pt",map_location=DEVICE,weights_only=True)["state_dict"])
    mlp.eval()
    for index, record in enumerate(tqdm(validation_pairs,desc=f"generate {n_tokens}")):
        raw = torch.from_numpy(np.array(val_features[index:index+1],copy=True)).to(DEVICE)
        with torch.inference_mode(), torch.autocast("cuda",dtype=torch.float16):
            visual = mlp(merge_patches(raw,n_tokens)).to(torch.float16)
            prompt = llm.get_input_embeddings()(torch.tensor([prompt_ids],device=DEVICE))
            prefix = torch.cat([visual,prompt],dim=1)
            generated = llm.generate(inputs_embeds=prefix,attention_mask=torch.ones(prefix.shape[:2],device=DEVICE,dtype=torch.long),
                                     max_new_tokens=48,do_sample=False,pad_token_id=tokenizer.pad_token_id,
                                     eos_token_id=tokenizer.eos_token_id,use_cache=True)
        caption = tokenizer.decode(generated[0],skip_special_tokens=True).strip()
        generation_rows.append({"visual_tokens":n_tokens,"image_id":record["image_id"],"caption":caption,
                                "references":record["references"]})
        save_json("generations.json",generation_rows)
    del mlp

Метрика — corpus BLEU-4 из `pycocoevalcap` с PTBTokenizer и всеми референсами. В Colab нужна Java; ячейка ниже проверяет её наличие. BLEU не является оценкой фактической правильности, поэтому ошибки разбираются отдельно по изображениям.

In [ ]:
import shutil
import subprocess
if shutil.which("java") is None:
    if Path("/content").exists():
        subprocess.run(["apt-get","update","-qq"],check=True)
        subprocess.run(["apt-get","install","-y","-qq","default-jre-headless"],check=True)
    else:
        raise RuntimeError("Для официального PTBTokenizer установите Java и добавьте java в PATH")
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
metric_rows = []
ptb = PTBTokenizer()
for n_tokens in [196,49]:
    rows = [r for r in generation_rows if r["visual_tokens"] == n_tokens]
    assert len(rows) == 500
    refs = {r["image_id"]:[{"caption":s} for s in r["references"]] for r in rows}
    hypotheses = {r["image_id"]:[{"caption":r["caption"]}] for r in rows}
    bleu, per_image = Bleu(4).compute_score(ptb.tokenize(refs),ptb.tokenize(hypotheses))
    for row, score in zip(rows,per_image[3]):
        row["BLEU4_individual"] = float(score)
    metric_rows.append({"visual_tokens":n_tokens,"BLEU4":float(bleu[3]),"n":len(rows)})
save_json("generations.json",generation_rows)
metrics = pd.DataFrame(metric_rows).merge(pd.read_csv(OUT/"timing.csv"),on="visual_tokens")
metrics.to_csv(OUT/"metrics.csv",index=False)
display(metrics)

## 6. Полученные результаты

In [ ]:
training = pd.read_csv(OUT/"training.csv")
plt.figure(figsize=(9,4))
for n_tokens, group in training.groupby("visual_tokens"):
    plt.plot(np.arange(len(group)),group.loss.rolling(50,min_periods=1).mean(),label=f"{n_tokens} tokens")
plt.xlabel("Microbatch")
plt.ylabel("Training loss, moving average 50")
plt.legend()
save_figure("training_curves.png")

## 7. Анализ и выводы
Ниже выбираются пять низких и пять высоких индивидуальных BLEU-4 для варианта 196; для тех же десяти изображений показываются обе генерации. Это отбор для разбора, а не случайная оценка частот ошибок. Низкий BLEU не назначает тип ошибки автоматически. При просмотре изображения можно указать несколько типов: object, count, attribute; ok — ошибок этих типов нет. Ответы сохраняются и при повторном запуске используются снова.

In [ ]:
base = sorted([r for r in generation_rows if r["visual_tokens"]==196],key=lambda r:r["BLEU4_individual"])
selected_ids = [r["image_id"] for r in base[:5]+base[-5:]]
review_path = OUT/"error_review.json"
reviews = json.loads(review_path.read_text(encoding="utf-8")) if review_path.exists() else []
finished = {(r["image_id"],r["visual_tokens"]) for r in reviews}
for image_id in selected_ids:
    record = next(r for r in validation_pairs if r["image_id"] == image_id)
    display(coco_image(record).resize((480,360)))
    display(Markdown("Эталоны: " + " | ".join(record["references"])))
    for row in [r for r in generation_rows if r["image_id"] == image_id]:
        display(Markdown(f"**{row['visual_tokens']} токенов:** {row['caption']}"))
        if (image_id,row["visual_tokens"]) not in finished:
            while True:
                answer = input("Типы ошибок через запятую: object, count, attribute или ok: ").strip().lower()
                labels = sorted(set(t.strip() for t in answer.split(",")))
                if set(labels) <= {"object","count","attribute","ok"} and labels != [""] and ("ok" not in labels or len(labels)==1):
                    break
            reason = input("Краткое обоснование по изображению: ").strip()
            while not reason:
                reason = input("Нужно непустое обоснование: ").strip()
            reviews.append({"image_id":image_id,"visual_tokens":row["visual_tokens"],"labels":labels,"reason":reason})
            save_json("error_review.json",reviews)
display(pd.DataFrame(reviews))

## 8. Контрольные вопросы
**Почему LLM замораживается?** На этапе согласования учится отображение визуальных признаков в пространство уже обученной языковой модели. Это уменьшает число обучаемых параметров и риск забывания языка. Градиент всё равно проходит через LLM до проектора.

**Что ожидается после этапа 1?** Простые подписи к изображениям обучающего распределения. Надёжное выполнение визуальных инструкций, диалог, точный счёт и отсутствие галлюцинаций этим этапом не обеспечиваются.

In [7]:
unit = json.loads((OUT/"unit_tests.json").read_text(encoding="utf-8"))
findings = [f"Проверена сборка labels, padding и причинный сдвиг. Loss маленькой случайной Qwen: {unit['causal_loss']:.6f}; ручной cross-entropy: {unit['manual_loss']:.6f}. Это тест реализации, а не обучение на COCO."]
if (OUT/"metrics.csv").exists():
    table = pd.read_csv(OUT/"metrics.csv")[["visual_tokens","BLEU4","seconds","max_context"]]
    findings.append("Оба варианта оценены на 500 изображениях. Сопоставляются BLEU-4, время обучения без кэширования признаков и максимальная длина контекста.")
    findings.append("Разбор ошибок завершён." if (OUT/"error_review.json").exists() and len(json.loads((OUT/"error_review.json").read_text(encoding="utf-8")))==20 else "Визуальная классификация ошибок ещё не завершена.")
else:
    table = pd.DataFrame([{"test":"causal LM loss","value":unit["causal_loss"]},{"test":"manual shifted loss","value":unit["manual_loss"]},{"test":"visual gradient norm","value":unit["visual_gradient_norm"]}])
    findings += ["Обучение двух проекторов на 20000 парах и генерация на 500 изображениях не выполнены: CUDA GPU в доступной среде нет. Значения BLEU-4, время обучения и реальные примеры генерации отсутствуют.",
                 "Полный цикл реализован для T4: fp16, AdamW, warmup, одна эпоха, кэш SigLIP, 196 и 49 токенов. Код библиотеки BLEU-4 и сохранение ручной классификации ошибок включены.",
                 "Теоретически объединение патчей уменьшает визуальную часть контекста в четыре раза. Влияние на качество и реальную скорость без запуска не установлено."]
write_report("ПЗ-3. Мини-LLaVA", "Сравнить проекторы с 196 и 49 визуальными токенами.",
             "Реализованы сборка inputs_embeds, маски, обучение и оценка. CPU-тест проверяет также прохождение градиента через замороженную LLM.",findings,table,
             OUT/"training_curves.png")

PDF: C:\Users\darri\PycharmProjects\cv061126\PZ_3\report.pdf
